# Fock-PARFLM v2.1 Gamma Sweep with Geodesic Residual Analysis — TinyStories

Systematic sweep of the damping coefficient $\gamma$ on TinyStories using
the **Fock v2.1 creation-gate routing fix** model (same architecture as
`colab_fock_v21_routing_fix.ipynb`, arm `v21_tau_perK_ortho`).

## Goals

1. Find the optimal $\gamma$ for TinyStories by training one short run per candidate.
2. Compute the **geodesic residual** $\bar{R}(\gamma)$ on the best checkpoint
   from each $\gamma$ to quantify how well the learned hidden-state trajectory
   follows the Jacobi geodesic of the conformal Maupertuis metric.
3. Overlay $\text{PPL}(\gamma)$ and $\bar{R}(\gamma)$ to identify the damping
   regime that best combines language-modelling quality with geodesic fidelity.

## Two-stage causal probes

- **Stage 1 — architectural probe** (CPU, float64, tiny model): verifies that
  the `prefix_causal_registers` fix produces exactly zero future sensitivity.
- **Stage 2 — trained-scale leak probe** (GPU, float32, live model): measures
  future-perturbation leakage and honest PPL on the actual trained weights.

## Companion documents

- `companion_notes/Geodesic_Preservation_Experiment.md` — theory and design
- `companion_notes/Fock-PARFLM_Causal_Leak_Audit_Results.md` — causal leak analysis

In [ ]:
# ── Cell 0: Sweep Configuration ────────────────────────────────────

GAMMA_CANDIDATES = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]
SWEEP_STEPS      = 3_000
SWEEP_EVAL_INTERVAL = 500

N_GEODESIC_BATCHES = 10
GEODESIC_SEED      = 42
GEODESIC_EPSILON   = 1e-6

# ── Architecture (matches colab_fock_v21_routing_fix.ipynb v21_tau_perK_ortho) ──
D              = 256
L              = 8
N_REGISTERS    = 16
BLOCK_SIZE     = 512
VOCAB_SIZE     = 50257

V_HIDDEN       = 1024
V_DEPTH        = 3

V_PHI_KIND     = 'structural_competitive'
V_PHI_PHI_HIDDEN  = 128
V_PHI_THETA_HIDDEN = 128
V_PHI_D_TYPE   = 32
V_PHI_D_ANGLE  = 16
TOP_K          = 8
V_PHI_N_HEADS  = 1    # default for scaleup mode

SCORE_HEAD_HIDDEN = 32
GUMBEL_TAU_INIT  = 1.0
GUMBEL_TAU_MIN   = 0.3

REVERSE_CHANNEL              = True
REVERSE_CHANNEL_STABLE       = False   # default for scaleup mode
REVERSE_CHANNEL_PRE_LN       = False
REVERSE_CHANNEL_SOFT_NORM    = False
REVERSE_CHANNEL_WARMUP_STEPS = 0
REVERSE_CHANNEL_PER_LAYER    = False

REGISTER_REPULSION       = False
REGISTER_REPULSION_COEFF = 0.0

USE_OUTPUT_BIAS = False   # default
TIE_EMBEDDINGS  = True    # default

# v2.1 creation-gate fixes
PER_REGISTER_TAU   = True
PER_REGISTER_KEYS  = True
ORTHO_REGISTER_INIT = True
TAU_CREATE_INIT    = 8.0   # sqrt(d_k=64)
D_K                = 64

XI_CHANNELS    = 4
XI_ALPHA_INIT_MODE = 'log_spaced'
XI_ALPHA_INITS = [0.0] * XI_CHANNELS   # overridden by log_spaced

LR             = 5e-4
WEIGHT_DECAY   = 0.01
LAMBDA_V       = 0.0
BATCH_SIZE     = 4
GRAD_ACCUM     = 4    # effective batch = 16 (same as routing fix)
GRAD_CLIP      = 1.0
FOCK_GRAD_CLIP = 0.5

# ── Causal probe intervals ──
CAUSAL_PROBE_INTERVAL       = 2000   # architectural probe every N steps (0 = off)
TRAINED_LEAK_PROBE_INTERVAL = 0      # trained-scale probe (0 = off for short sweep)
TRAINED_LEAK_PROBE_K        = 256
TRAINED_LEAK_PROBE_PAIRS    = 2

MAX_TRAIN_TOKENS = 5_000_000

print(f'Gamma sweep: {GAMMA_CANDIDATES}')
print(f'Steps per candidate: {SWEEP_STEPS:,}')
print(f'd={D}  L={L}  M={N_REGISTERS}  batch={BATCH_SIZE}x{GRAD_ACCUM} (eff={BATCH_SIZE*GRAD_ACCUM})')
print(f'Geodesic analysis: {N_GEODESIC_BATCHES} validation batches, seed={GEODESIC_SEED}')

In [ ]:
# ── Cell 1: Environment + Drive Mount ──────────────────────────────
import os, sys, gc, shutil, subprocess, json, time, math, copy
from pathlib import Path

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

REPO_URL    = 'https://github.com/dimitarpg13/semsimula-paper.git'
REPO_BRANCH = 'main'

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')


def _sh(cmd):
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'exit {r.returncode}: {cmd}')


if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    REPO_ROOT = Path('/content/semsimula-paper')
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(f'git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_ROOT}')
    else:
        try:
            _sh(f'git -C {REPO_ROOT} fetch --depth 1 origin {REPO_BRANCH}')
            _sh(f'git -C {REPO_ROOT} reset --hard origin/{REPO_BRANCH}')
        except RuntimeError as e:
            print(f'WARNING: repo refresh failed ({e}); using existing checkout.')

    _sh('pip install -q transformers huggingface_hub pyarrow matplotlib')

    GDRIVE_ROOT = Path('/content/drive/MyDrive/semsimula_fock_gamma_sweep_geodesic_tinystories')
    GDRIVE_ROOT.mkdir(parents=True, exist_ok=True)

    DATA_DIR = GDRIVE_ROOT / 'data'
    DATA_DIR.mkdir(exist_ok=True)

    repo_data = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    if repo_data.is_symlink():
        repo_data.unlink()
    elif repo_data.is_dir():
        shutil.rmtree(repo_data)
    repo_data.symlink_to(DATA_DIR)

    SWEEP_OUTPUT_DIR = GDRIVE_ROOT / 'gamma_sweep'
    RESULTS_DIR      = GDRIVE_ROOT / 'results'
    SWEEP_OUTPUT_DIR.mkdir(exist_ok=True)
    RESULTS_DIR.mkdir(exist_ok=True)
else:
    REPO_ROOT = Path('.').resolve()
    while not (REPO_ROOT / '.git').exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent
    DATA_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    SWEEP_OUTPUT_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup' / 'results' / 'gamma_sweep_geodesic_tinystories'
    RESULTS_DIR = SWEEP_OUTPUT_DIR
    for d in [DATA_DIR, SWEEP_OUTPUT_DIR, RESULTS_DIR]:
        d.mkdir(parents=True, exist_ok=True)

CA_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch'
for sub in ['', 'parf', 'multixi', 'scaleup', 'sarf_mass_variant', 'energetic_minima']:
    d = str(CA_DIR / sub) if sub else str(CA_DIR)
    if d not in sys.path:
        sys.path.insert(0, d)

print(f'DATA_DIR         = {DATA_DIR}')
print(f'SWEEP_OUTPUT_DIR = {SWEEP_OUTPUT_DIR}')
print(f'RESULTS_DIR      = {RESULTS_DIR}')

In [ ]:
# ── Cell 2: GPU Check + Imports ────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}  ({props.total_memory / 1e9:.1f} GB)')
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    print('TF32 disabled for PARF autograd.grad stability')
else:
    print('WARNING: No GPU detected. This notebook requires CUDA.')

from model_fock_parf_multixi import FockMultiXiPARFLM, FockMultiXiPARFConfig
import model_fock_parf_v2
import model_parf_multixi
import model_parf
import model_parf_sparse

print('Model imports OK')

In [ ]:
# ── Cell 3: Data Loading (TinyStories) ─────────────────────────────
from data_module import get_batch, load_tiny_stories

train_ids, val_ids = load_tiny_stories(
    n_train_files=1,
    val_frac=0.01,
    max_train_tokens=MAX_TRAIN_TOKENS,
)

print(f'train: {len(train_ids):,}   val: {len(val_ids):,}')

In [ ]:
# ── Cell 4: Logfreq Surprisal ─────────────────────────────────────
LOGFREQ_PATH = CA_DIR / 'scaleup' / 'results' / 'logfreq_surprisal_tinystories.npy'
DRIVE_LOGFREQ = RESULTS_DIR / 'logfreq_surprisal_tinystories.npy'

if LOGFREQ_PATH.exists():
    LOGFREQ_FILE = LOGFREQ_PATH
elif DRIVE_LOGFREQ.exists():
    LOGFREQ_FILE = DRIVE_LOGFREQ
else:
    counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE).astype(np.float64)
    p = (counts + 1.0) / (counts.sum() + VOCAB_SIZE)
    surprisal = (-np.log(p)).astype(np.float32)
    np.save(str(DRIVE_LOGFREQ), surprisal)
    LOGFREQ_FILE = DRIVE_LOGFREQ
    print(f'  Computed logfreq surprisal -> {LOGFREQ_FILE}')

print(f'Logfreq: {LOGFREQ_FILE}')

In [ ]:
# ── Cell 5: Model Builder ─────────────────────────────────────────

def build_model(gamma, device=DEVICE):
    """Build a Fock v2.1 PARFLM model matching colab_fock_v21_routing_fix.ipynb."""
    model_cfg = FockMultiXiPARFConfig(
        vocab_size=VOCAB_SIZE,
        d=D,
        max_len=1024,
        L=L,
        v_hidden=V_HIDDEN,
        v_depth=V_DEPTH,
        v_phi_d_type=V_PHI_D_TYPE,
        v_phi_d_angle=V_PHI_D_ANGLE,
        mass_mode='logfreq',
        logfreq_path=str(LOGFREQ_FILE),
        logfreq_init_alpha=0.1,
        init_gamma=1.0,
        fixed_gamma=gamma,
        causal_force=True,
        ln_after_step=True,
        xi_channels=XI_CHANNELS,
        xi_alpha_inits=XI_ALPHA_INITS,
        xi_learnable=True,
        xi_alpha_init_mode=XI_ALPHA_INIT_MODE,
        v_phi_kind=V_PHI_KIND,
        v_phi_phi_hidden=V_PHI_PHI_HIDDEN,
        v_phi_theta_hidden=V_PHI_THETA_HIDDEN,
        top_k=TOP_K,
        v_phi_n_heads=V_PHI_N_HEADS,
        score_head_hidden=SCORE_HEAD_HIDDEN,
        gumbel_tau_init=GUMBEL_TAU_INIT,
        gumbel_tau_min=GUMBEL_TAU_MIN,
        gumbel_noise=True,
        use_gathered_v_phi=True,
        use_layer_checkpoint=True,
        ln_before_distance=True,
        per_layer_v_phi_scale=True,
        use_output_bias=USE_OUTPUT_BIAS,
        tie_embeddings=TIE_EMBEDDINGS,
        fock_version='v2',
        n_registers=N_REGISTERS,
        reverse_channel=REVERSE_CHANNEL,
        reverse_channel_stable=REVERSE_CHANNEL_STABLE,
        reverse_channel_pre_ln=REVERSE_CHANNEL_PRE_LN,
        reverse_channel_soft_norm=REVERSE_CHANNEL_SOFT_NORM,
        reverse_channel_warmup_steps=REVERSE_CHANNEL_WARMUP_STEPS,
        reverse_channel_per_layer=REVERSE_CHANNEL_PER_LAYER,
        register_repulsion=REGISTER_REPULSION,
        register_repulsion_coeff=REGISTER_REPULSION_COEFF,
        d_k=D_K,
        tau_create_init=TAU_CREATE_INIT,
        per_register_tau=PER_REGISTER_TAU,
        per_register_keys=PER_REGISTER_KEYS,
        ortho_register_init=ORTHO_REGISTER_INIT,
        stack_discipline=True,
        prefix_causal_registers=True,
    )
    model = FockMultiXiPARFLM(model_cfg).to(device)

    n_params = sum(p.numel() for p in model.parameters())
    print(f'  Model built: gamma={gamma:.3f}  params={n_params:,}')
    return model, model_cfg


_test_model, _test_cfg = build_model(0.30)
del _test_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Model builder OK')

In [ ]:
# ── Cell 6: Training + Evaluation Helpers ──────────────────────────

def forward_fock(model, x, targets):
    """Forward pass returning (loss, ntp_loss)."""
    h0 = model._embed(x)
    h_L, _ = model._stack_forward(h0, x, return_trajectory=False)
    logits = model.compute_logits(h_L)
    loss_ntp = F.cross_entropy(
        logits.float().reshape(-1, VOCAB_SIZE),
        targets.reshape(-1),
    )
    return loss_ntp, loss_ntp


@torch.no_grad()
def evaluate(model, val_ids, n_iters=40):
    """Evaluate on validation set. Returns (val_loss, val_ppl)."""
    model.eval()
    rng = np.random.default_rng(42)
    losses = []
    for _ in range(n_iters):
        x_np, y_np = get_batch(val_ids, BATCH_SIZE, BLOCK_SIZE, rng)
        x = torch.from_numpy(x_np).to(DEVICE)
        y = torch.from_numpy(y_np).to(DEVICE)
        with torch.enable_grad():
            _, ntp = forward_fock(model, x, y)
        losses.append(ntp.item())
    model.train()
    val_loss = sum(losses) / len(losses)
    val_ppl = math.exp(val_loss)
    return val_loss, val_ppl


def get_wsd_lr(step, total_steps, peak_lr, floor_lr=None):
    """WSD learning rate schedule."""
    if floor_lr is None:
        floor_lr = peak_lr * 0.05
    warmup_steps = int(total_steps * 0.05)
    stable_end   = int(total_steps * 0.65)
    if step < warmup_steps:
        return peak_lr * (step + 1) / warmup_steps
    elif step < stable_end:
        return peak_lr
    else:
        decay_steps = total_steps - stable_end
        progress = (step - stable_end) / max(decay_steps, 1)
        return floor_lr + 0.5 * (peak_lr - floor_lr) * (1 + math.cos(math.pi * progress))


print('Helpers OK')

In [ ]:
# ── Cell 7: Per-Group Gradient Clipping ────────────────────────────

FOCK_LAYER_PREFIX = 'fock_layers.'
VPHI_PREFIX       = 'V_phi.'
VTHETA_PREFIX     = 'V_theta.'

REVERSE_CH_EXCLUDE = {'reverse_channel_scale'}

CLIP_LIMITS = {
    'fock':   FOCK_GRAD_CLIP,
    'vphi':   GRAD_CLIP * 0.3,
    'vtheta': GRAD_CLIP,
    'other':  GRAD_CLIP,
}


def _classify(name):
    if name.startswith(FOCK_LAYER_PREFIX):
        return 'fock'
    elif name.startswith(VPHI_PREFIX):
        return 'vphi'
    elif name.startswith(VTHETA_PREFIX):
        return 'vtheta'
    elif any(k in name for k in REVERSE_CH_EXCLUDE):
        return 'reverse_channel_scale'
    else:
        return 'other'


def clip_grad_per_group(model):
    """Per-group gradient clipping. Returns (total_norm, top_group, top_norm)."""
    groups = {}
    for name, p in model.named_parameters():
        if p.grad is None:
            continue
        g = _classify(name)
        groups.setdefault(g, []).append(p)

    group_norms = {}
    for g, params in groups.items():
        gnorm = torch.nn.utils.clip_grad_norm_(params, CLIP_LIMITS.get(g, GRAD_CLIP))
        group_norms[g] = float(gnorm)

    all_params = [p for p in model.parameters() if p.grad is not None]
    total_norm = sum(p.grad.data.norm().item() ** 2 for p in all_params) ** 0.5

    filtered = {k: v for k, v in group_norms.items() if k not in REVERSE_CH_EXCLUDE}
    top_group = max(filtered, key=filtered.get) if filtered else 'none'
    top_norm = filtered.get(top_group, 0.0)

    return total_norm, top_group, top_norm


print('Per-group clipping OK')

In [ ]:
# ── Cell 8: Geodesic Residual Functions ────────────────────────────
from typing import Dict, List, Tuple


def collect_trajectory(model, x):
    """Forward pass returning per-layer hidden states [h_0, ..., h_L]."""
    with torch.enable_grad():
        h0 = model._embed(x)
        _, traj = model._stack_forward(h0, x, return_trajectory=True)
    return traj


def vtheta_value_and_grad(model, h_ell, layer_idx):
    """Evaluate V_theta and gradient at a given layer.

    Uses analytical_grad when available (structured / Gaussian V_theta),
    otherwise falls back to torch.autograd.grad (MLP V_theta).
    """
    xis = model.xi_module(h_ell)
    if hasattr(model.V_theta, "set_active_layer"):
        model.V_theta.set_active_layer(layer_idx)

    if hasattr(model.V_theta, "analytical_grad"):
        with torch.no_grad():
            V = model.V_theta(xis, h_ell).squeeze(-1)
            grad_V = model.V_theta.analytical_grad(xis, h_ell)
    else:
        with torch.enable_grad():
            h_ag = h_ell.detach().requires_grad_(True)
            xis_ag = xis.detach().requires_grad_(True)
            V_ag = model.V_theta(xis_ag, h_ag).squeeze(-1)
            V_sum = V_ag.sum()
            grad_h, = torch.autograd.grad(V_sum, h_ag, create_graph=False)
        V = V_ag.detach()
        grad_V = grad_h.detach()
    return V, grad_V


def conformal_grad(grad_V, E_minus_V, epsilon=1e-6):
    """Gradient of the Jacobi conformal factor phi = 0.5 * log(2*(E-V))."""
    denom = 2.0 * E_minus_V.unsqueeze(-1).clamp(min=epsilon)
    return -grad_V / denom


def christoffel_vv(phi_grad, v):
    """Christoffel-velocity contraction for conformally flat metric."""
    phi_dot_v = (phi_grad * v).sum(dim=-1, keepdim=True)
    v_sq = (v * v).sum(dim=-1, keepdim=True)
    return 2.0 * phi_dot_v * v - v_sq * phi_grad


@torch.no_grad()
def compute_residual(traj, model, gamma_eval, device, epsilon=1e-6, ref_layer=0):
    """Compute the damped-geodesic residual R_bar for one trajectory."""
    L = len(traj) - 1

    h_ref = traj[ref_layer].to(device)
    v_ref = traj[ref_layer + 1].to(device) - h_ref
    KE_ref = 0.5 * (v_ref * v_ref).sum(dim=-1)
    V_ref, _ = vtheta_value_and_grad(model, h_ref, ref_layer)
    E = KE_ref + V_ref
    del h_ref, v_ref, KE_ref, V_ref

    per_layer_R = []
    total_excluded = 0
    total_tokens = 0
    num_sum = 0.0
    den_sum = 0.0

    for ell in range(1, L):
        h_prev = traj[ell - 1].to(device)
        h_curr = traj[ell].to(device)
        h_next = traj[ell + 1].to(device)

        v_ell = h_curr - h_prev
        v_next = h_next - h_curr
        a_ell = v_next - v_ell

        V_ell, grad_V_ell = vtheta_value_and_grad(model, h_curr, ell)
        E_minus_V = E - V_ell
        allowed = E_minus_V > epsilon
        total_excluded += int((~allowed).sum().item())
        total_tokens += allowed.numel()

        phi_g = conformal_grad(grad_V_ell, E_minus_V, epsilon)
        Gamma_vv = christoffel_vv(phi_g, v_ell)

        residual_vec = a_ell + Gamma_vv + gamma_eval * v_ell
        residual_norm = residual_vec.norm(dim=-1)
        a_norm = a_ell.norm(dim=-1)
        R_ell = residual_norm / (a_norm + epsilon)

        R_ell = R_ell * allowed.float()
        n_allowed = allowed.float().sum().clamp(min=1.0)
        mean_R = R_ell.sum() / n_allowed
        per_layer_R.append(float(mean_R.item()))

        a_plus_Gamma = a_ell + Gamma_vv
        dot_num = (a_plus_Gamma * v_ell).sum(dim=-1)
        v_sq = (v_ell * v_ell).sum(dim=-1)
        num_sum += float((dot_num * allowed.float()).sum().item())
        den_sum += float((v_sq * allowed.float()).sum().item())

        del h_prev, h_curr, h_next, v_ell, v_next, a_ell
        del V_ell, grad_V_ell, phi_g, Gamma_vv

    R_bar = float(np.mean(per_layer_R)) if per_layer_R else float('inf')
    excluded_frac = total_excluded / max(total_tokens, 1)
    gamma_geo = -num_sum / max(den_sum, 1e-12)

    return {
        'R_bar': R_bar,
        'per_layer_R': per_layer_R,
        'excluded_frac': excluded_frac,
        'gamma_geo': gamma_geo,
    }


print('Geodesic residual functions OK')

In [ ]:
# ── Cell 9: Two-Stage Causal Probes ──────────────────────────────

def run_causal_probe(step_num, model_cfg_ref):
    """Stage 1: lightweight architectural causal probe (CPU, float64).

    Builds a tiny model with the same structural features, opens the
    reverse channel fully, and checks that perturbing future tokens
    produces exactly zero change in logits at earlier positions.
    Returns (passed: bool, max_delta: float).
    """
    _PROBE_VOCAB, _PROBE_D, _PROBE_L = 101, 32, 4
    _PROBE_T, _PROBE_M, _PROBE_XI = 48, 8, 3

    _logfreq_probe = Path('/tmp/causal_probe_logfreq.npy')
    np.save(_logfreq_probe, np.full(_PROBE_VOCAB, 5.0, dtype=np.float32))

    _probe_cfg = FockMultiXiPARFConfig(
        vocab_size=_PROBE_VOCAB, d=_PROBE_D, max_len=64, L=_PROBE_L,
        v_hidden=64, v_depth=1, dt=0.1,
        mass_mode='global', causal_force=True, ln_after_step=True,
        xi_channels=_PROBE_XI, xi_alpha_inits=[0.5, 0.9, 0.99],
        xi_learnable=False, xi_alpha_init_mode='explicit',
        fock_version='v2', n_registers=_PROBE_M,
        register_salience_decay=0.5, register_salience_threshold=0.005,
        creation_gate_hidden=16, stack_discipline=True,
        d_k=16, tau_create_init=8.0,
        reverse_channel=True,
        per_register_tau=True, per_register_keys=True,
        ortho_register_init=True,
        prefix_causal_registers=True,
    )
    torch.manual_seed(1234)
    _probe_model = FockMultiXiPARFLM(_probe_cfg).double().cpu()

    with torch.no_grad():
        for n, p in _probe_model.named_parameters():
            if 'reverse_channel_scale' in n:
                p.fill_(5.0)

    _t_p = _PROBE_T // 2
    _prng = np.random.default_rng(7)
    _x1 = torch.from_numpy(_prng.integers(0, _PROBE_VOCAB, (2, _PROBE_T))).long()
    _x2 = _x1.clone()
    _x2[:, _t_p:] = torch.from_numpy(
        _prng.integers(0, _PROBE_VOCAB, (2, _PROBE_T - _t_p))).long()

    _max_delta = 0.0
    for mode_name, use_train in [('eval', False), ('train', True)]:
        if use_train:
            _probe_model.train()
            torch.manual_seed(99)
        else:
            _probe_model.eval()
        with torch.enable_grad():
            _la = _probe_model(_x1)[0].detach()
        if use_train:
            torch.manual_seed(99)
        with torch.enable_grad():
            _lb = _probe_model(_x2)[0].detach()
        delta = float((_la[:, :_t_p] - _lb[:, :_t_p]).abs().max().item())
        _max_delta = max(_max_delta, delta)

    _passed = (_max_delta == 0.0)

    del _probe_model, _la, _lb, _x1, _x2
    gc.collect()

    status = 'PASS' if _passed else '*** FAIL ***'
    print(f'\n[causal probe] step {step_num:,}  max|dlogit|={_max_delta:.3e}  [{status}]')
    if not _passed:
        print('[causal probe] WARNING: nonzero future sensitivity detected!')
    return _passed, _max_delta


def run_trained_leak_probe(step_num, model, val_ids_arr):
    """Stage 2: trained-scale leak probe + honest PPL (live trained model).

    Returns a dict with probe and honest-PPL results.
    """
    _debug_dir = str(CA_DIR / 'scaleup' / 'debug')
    if _debug_dir not in sys.path:
        sys.path.insert(0, _debug_dir)
    from fock_trained_leak_probe import probe_trained_leak, honest_ppl_test

    print(f'\n{"="*64}')
    print(f'[trained leak probe] step {step_num:,} — running on live model')
    print(f'{"="*64}')

    probe_res = probe_trained_leak(
        model, val_ids_arr, device=DEVICE, context=BLOCK_SIZE,
        n_pairs=TRAINED_LEAK_PROBE_PAIRS, use_float64=False)

    honest_res = honest_ppl_test(
        model, val_ids_arr, k=TRAINED_LEAK_PROBE_K,
        context=BLOCK_SIZE, batch=BATCH_SIZE, device=DEVICE)

    model.train()

    result = {
        'step': step_num,
        'event': 'trained_leak_probe',
        'probe_max_dlogit_past': probe_res['max_dlogit_past'],
        'probe_mean_dnll_past_nats': round(probe_res['mean_dnll_past'], 6),
        'probe_gate_zero_control': probe_res['gate_zero_control'],
        'honest_k': honest_res['k'],
        'ppl_mid_window_standard': round(honest_res['ppl_mid_window'], 4),
        'ppl_last_pos_leak_free': round(honest_res['ppl_last_pos'], 4),
        'paired_diff_nats': round(honest_res['paired_diff_nats'], 6),
        'paired_diff_se': round(honest_res['paired_diff_se'], 6),
    }

    _leak_status = 'CLEAN' if result['paired_diff_nats'] < 0.1 else 'LEAK DETECTED'
    print(f'\n[trained leak probe] step {step_num:,}  '
          f'honest_PPL={result["ppl_last_pos_leak_free"]:.2f}  '
          f'standard_PPL={result["ppl_mid_window_standard"]:.2f}  '
          f'diff={result["paired_diff_nats"]:+.4f} nats  [{_leak_status}]')
    return result


print('Two-stage causal probes OK')

In [ ]:
# ── Cell 10: Single-Gamma Training Function ──────────────────────

def train_one_gamma(gamma, sweep_dir, train_ids, val_ids):
    """Train one gamma candidate and save checkpoint. Returns result dict.

    Resume-aware: if ckpt_best.pt exists but its stored step < SWEEP_STEPS,
    the run is treated as incomplete — model + optimizer are restored and
    training continues from the next step.  Only a checkpoint whose step
    >= SWEEP_STEPS is considered truly finished.
    """
    gamma_dir = sweep_dir / f'gamma_{gamma:.3f}'
    ckpt_dir  = gamma_dir / 'checkpoints'
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    log_path  = gamma_dir / 'training_log.jsonl'

    best_ckpt = ckpt_dir / 'ckpt_best.pt'
    resume_step = 0
    resume_data = None

    if best_ckpt.exists():
        ckpt = torch.load(str(best_ckpt), map_location='cpu', weights_only=False)
        ckpt_step = ckpt.get('step', 0)
        best_ppl_loaded = ckpt.get('val_ppl', float('inf'))

        if ckpt_step >= SWEEP_STEPS:
            print(f'  [SKIP] gamma={gamma:.3f} already complete '
                  f'(step {ckpt_step:,}/{SWEEP_STEPS:,}) — best PPL={best_ppl_loaded:.2f}')
            del ckpt
            return {
                'gamma': gamma,
                'best_ppl': best_ppl_loaded,
                'skipped': True,
            }
        else:
            print(f'  [RESUME] gamma={gamma:.3f} incomplete '
                  f'(step {ckpt_step:,}/{SWEEP_STEPS:,}, PPL={best_ppl_loaded:.2f}) '
                  f'— continuing for {SWEEP_STEPS - ckpt_step:,} more steps')
            resume_step = ckpt_step
            resume_data = ckpt

    print(f'\n{"="*60}')
    print(f'  GAMMA = {gamma:.3f}  ({SWEEP_STEPS:,} steps)')
    print(f'{"="*60}')

    model, model_cfg = build_model(gamma)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR, betas=(0.9, 0.95),
        weight_decay=WEIGHT_DECAY)

    best_ppl = float('inf')
    best_loss = float('inf')

    if resume_data is not None:
        model.load_state_dict(resume_data['model_state_dict'], strict=False)
        if 'optimizer_state_dict' in resume_data:
            try:
                optimizer.load_state_dict(resume_data['optimizer_state_dict'])
                print(f'  Optimizer state restored.')
            except (ValueError, KeyError) as e:
                print(f'  [info] Optimizer state incompatible, starting fresh: {e}')
        best_ppl = resume_data.get('val_ppl', float('inf'))
        best_loss = resume_data.get('val_loss', float('inf'))
        del resume_data

    model.train()

    rng = np.random.default_rng(0)
    # Advance the RNG past the steps already completed so the data ordering
    # is identical to what a single uninterrupted run would have seen.
    if resume_step > 0:
        for _ in range(resume_step * GRAD_ACCUM):
            get_batch(train_ids, BATCH_SIZE, BLOCK_SIZE, rng)

    t0 = time.time()

    log_entries = []
    if log_path.exists():
        with open(str(log_path)) as f:
            for line in f:
                try:
                    log_entries.append(json.loads(line))
                except json.JSONDecodeError:
                    pass

    for step in range(resume_step + 1, SWEEP_STEPS + 1):
        lr = get_wsd_lr(step, SWEEP_STEPS, LR)
        for pg in optimizer.param_groups:
            pg['lr'] = lr

        model.train()
        optimizer.zero_grad(set_to_none=True)

        accum_ntp = 0.0
        for _micro in range(GRAD_ACCUM):
            x_np, y_np = get_batch(train_ids, BATCH_SIZE, BLOCK_SIZE, rng)
            x = torch.from_numpy(x_np).to(DEVICE)
            y = torch.from_numpy(y_np).to(DEVICE)
            loss, ntp = forward_fock(model, x, y)
            if REGISTER_REPULSION:
                _rep = model.pop_repulsion_loss()
                loss = loss + _rep
            (loss / GRAD_ACCUM).backward()
            accum_ntp += ntp.item() / GRAD_ACCUM
            del loss, ntp, x, y

        total_norm, top_group, top_norm = clip_grad_per_group(model)
        optimizer.step()

        if step % 50 == 0:
            elapsed = time.time() - t0
            steps_done = step - resume_step
            remaining = elapsed / steps_done * (SWEEP_STEPS - step)
            print(f'  step {step:>5d}/{SWEEP_STEPS}  ntp={accum_ntp:.4f}  '
                  f'lr={lr:.2e}  grad={total_norm:.2f}  '
                  f'top[{top_group}]={top_norm:.1f}  '
                  f'{elapsed:.0f}s (~{remaining/3600:.1f}h remaining)')

        if step % SWEEP_EVAL_INTERVAL == 0 or step == SWEEP_STEPS:
            val_loss, val_ppl = evaluate(model, val_ids)
            is_best = val_ppl < best_ppl
            if is_best:
                best_ppl = val_ppl
                best_loss = val_loss
            # Always save with the current step so a future resume knows
            # how far we got; val_ppl/val_loss track the best, not latest.
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'step': step,
                'val_loss': best_loss,
                'val_ppl': best_ppl,
                'gamma': gamma,
            }, str(best_ckpt))
            marker = ' *** NEW BEST ***' if is_best else ''
            print(f'  >>> EVAL step {step:,}  val_loss={val_loss:.4f}  '
                  f'val_ppl={val_ppl:.2f}  best={best_ppl:.2f}{marker}')
            log_entries.append({
                'step': step, 'val_loss': val_loss, 'val_ppl': val_ppl,
                'best_ppl': best_ppl, 'train_ntp': accum_ntp,
            })

        # Stage 1: architectural causal probe
        if CAUSAL_PROBE_INTERVAL > 0 and step % CAUSAL_PROBE_INTERVAL == 0:
            _cp_passed, _cp_delta = run_causal_probe(step, model_cfg)
            log_entries.append({
                'step': step, 'event': 'causal_probe',
                'causal_probe_passed': _cp_passed,
                'causal_probe_max_delta': _cp_delta,
            })

        # Stage 2: trained-scale leak probe + honest PPL
        if TRAINED_LEAK_PROBE_INTERVAL > 0 and step % TRAINED_LEAK_PROBE_INTERVAL == 0:
            _tlp = run_trained_leak_probe(step, model, val_ids)
            log_entries.append(_tlp)

    elapsed = time.time() - t0
    print(f'  Done gamma={gamma:.3f}  best_ppl={best_ppl:.2f}  ({elapsed:.0f}s)')

    with open(str(log_path), 'w') as f:
        for entry in log_entries:
            f.write(json.dumps(entry) + '\n')

    del model, optimizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        'gamma': gamma,
        'best_ppl': best_ppl,
        'best_loss': best_loss,
        'wall_clock_s': elapsed,
        'skipped': False,
    }


print('Training function OK')

In [ ]:
# ── Cell 11: Run Gamma Sweep ──────────────────────────────────────

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

sweep_results = []

for gamma in GAMMA_CANDIDATES:
    result = train_one_gamma(gamma, SWEEP_OUTPUT_DIR, train_ids, val_ids)
    sweep_results.append(result)
    print(f'  gamma={gamma:.3f}  best_ppl={result["best_ppl"]:.2f}  '
          f'skipped={result.get("skipped", False)}')

print(f'\n{"="*60}')
print('Gamma sweep complete!')
print(f'{"="*60}')
for r in sweep_results:
    print(f'  gamma={r["gamma"]:.3f}  PPL={r["best_ppl"]:.2f}')

best_r = min(sweep_results, key=lambda x: x['best_ppl'])
print(f'\n  >>> Best: gamma={best_r["gamma"]:.3f}  PPL={best_r["best_ppl"]:.2f}')

with open(str(SWEEP_OUTPUT_DIR / 'sweep_summary.json'), 'w') as f:
    json.dump(sweep_results, f, indent=2)

print('Saved sweep_summary.json')

In [ ]:
# ── Cell 12: Geodesic Residual Analysis on Sweep Checkpoints ──────

geodesic_results = {}

rng_geo = np.random.default_rng(GEODESIC_SEED)

for gamma in GAMMA_CANDIDATES:
    gamma_dir = SWEEP_OUTPUT_DIR / f'gamma_{gamma:.3f}'
    ckpt_path = gamma_dir / 'checkpoints' / 'ckpt_best.pt'
    if not ckpt_path.exists():
        print(f'  gamma={gamma:.3f}: no checkpoint found, skipping geodesic analysis')
        continue

    print(f'\n--- gamma={gamma:.3f}: loading checkpoint for geodesic analysis ---')
    ckpt = torch.load(str(ckpt_path), map_location='cpu', weights_only=False)
    model, _ = build_model(gamma)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()

    all_R_bar = []
    all_gamma_geo = []
    all_excluded = []
    all_per_layer_R = []

    for bi in range(N_GEODESIC_BATCHES):
        x_np, _ = get_batch(val_ids, BATCH_SIZE, BLOCK_SIZE, rng_geo)
        x = torch.from_numpy(x_np).to(DEVICE)

        with torch.enable_grad():
            traj = collect_trajectory(model, x)

        res = compute_residual(
            traj, model, gamma, DEVICE,
            epsilon=GEODESIC_EPSILON,
        )
        all_R_bar.append(res['R_bar'])
        all_gamma_geo.append(res['gamma_geo'])
        all_excluded.append(res['excluded_frac'])
        all_per_layer_R.append(res['per_layer_R'])

        del traj, x
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    mean_R_bar = float(np.mean(all_R_bar))
    std_R_bar = float(np.std(all_R_bar))
    mean_gamma_geo = float(np.mean(all_gamma_geo))
    mean_excluded = float(np.mean(all_excluded))

    mean_per_layer = np.mean(all_per_layer_R, axis=0).tolist()

    geodesic_results[gamma] = {
        'R_bar_mean': mean_R_bar,
        'R_bar_std': std_R_bar,
        'gamma_geo_mean': mean_gamma_geo,
        'excluded_frac': mean_excluded,
        'per_layer_R': mean_per_layer,
        'val_ppl': ckpt.get('val_ppl', None),
    }

    print(f'  R_bar={mean_R_bar:.4f} +/- {std_R_bar:.4f}  '
          f'gamma_geo={mean_gamma_geo:.4f}  '
          f'excluded={mean_excluded:.2%}  '
          f'val_ppl={ckpt.get("val_ppl", "?")}' )

    del model, ckpt
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

with open(str(SWEEP_OUTPUT_DIR / 'geodesic_results.json'), 'w') as f:
    json.dump({str(k): v for k, v in geodesic_results.items()}, f, indent=2)

print(f'\nGeodesic analysis complete for {len(geodesic_results)} gammas.')

In [ ]:
# ── Cell 13: Overlay Plot — PPL(gamma) vs R_bar(gamma) ─────────────

gammas_plot = sorted(geodesic_results.keys())
ppls_plot   = [geodesic_results[g]['val_ppl'] for g in gammas_plot]
rbars_plot  = [geodesic_results[g]['R_bar_mean'] for g in gammas_plot]
rbar_stds   = [geodesic_results[g]['R_bar_std'] for g in gammas_plot]

fig, ax1 = plt.subplots(figsize=(10, 5))

color1 = '#1f77b4'
ax1.set_xlabel(r'$\gamma$ (damping coefficient)')
ax1.set_ylabel('Validation PPL', color=color1)
ax1.plot(gammas_plot, ppls_plot, 'o-', color=color1, linewidth=2, markersize=8,
         label='Val PPL')
ax1.tick_params(axis='y', labelcolor=color1)

ax2 = ax1.twinx()
color2 = '#d62728'
ax2.set_ylabel(r'$\bar{R}$ (geodesic residual)', color=color2)
ax2.errorbar(gammas_plot, rbars_plot, yerr=rbar_stds, fmt='s--', color=color2,
             linewidth=2, markersize=8, capsize=4, label=r'$\bar{R}$')
ax2.tick_params(axis='y', labelcolor=color2)

# Fitted gamma_geo line
gamma_geos = [geodesic_results[g]['gamma_geo_mean'] for g in gammas_plot]
for g_val, gg in zip(gammas_plot, gamma_geos):
    ax1.annotate(
        f'$\\hat{{\\gamma}}$={gg:.2f}',
        (g_val, ppls_plot[gammas_plot.index(g_val)]),
        textcoords='offset points', xytext=(0, 14),
        fontsize=7, ha='center', color='gray',
    )

best_idx = int(np.argmin(ppls_plot))
ax1.axvline(gammas_plot[best_idx], color='green', linestyle=':', alpha=0.5,
            label=f'Best PPL @ $\\gamma$={gammas_plot[best_idx]:.2f}')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.title('Fock v2.1 PARFLM — TinyStories Gamma Sweep\nPPL and Geodesic Residual')
fig.tight_layout()

plot_path = SWEEP_OUTPUT_DIR / 'ppl_vs_rbar_overlay_tinystories.png'
fig.savefig(str(plot_path), dpi=150)
print(f'Saved: {plot_path}')
plt.show()

# Per-layer R_bar heatmap
fig2, ax3 = plt.subplots(figsize=(10, 4))
layer_data = np.array([geodesic_results[g]['per_layer_R'] for g in gammas_plot])
im = ax3.imshow(layer_data, aspect='auto', cmap='viridis',
                extent=[0.5, layer_data.shape[1] + 0.5, len(gammas_plot) - 0.5, -0.5])
ax3.set_yticks(range(len(gammas_plot)))
ax3.set_yticklabels([f'{g:.2f}' for g in gammas_plot])
ax3.set_xlabel('Layer')
ax3.set_ylabel(r'$\gamma$')
ax3.set_title('Per-Layer Geodesic Residual — TinyStories')
fig2.colorbar(im, ax=ax3, label=r'$R_{\ell}$')
fig2.tight_layout()

heatmap_path = SWEEP_OUTPUT_DIR / 'per_layer_residual_heatmap_tinystories.png'
fig2.savefig(str(heatmap_path), dpi=150)
print(f'Saved: {heatmap_path}')
plt.show()

In [ ]:
# ── Cell 14: Final Summary ─────────────────────────────────────────

print(f'\n{"="*60}')
print('  GAMMA SWEEP + GEODESIC ANALYSIS  —  TinyStories')
print(f'{"="*60}\n')
print(f'Architecture: Fock v2.1 PARFLM  d={D}  L={L}  M={N_REGISTERS}')
print(f'Training:     {SWEEP_STEPS:,} steps  batch={BATCH_SIZE}x{GRAD_ACCUM}={BATCH_SIZE*GRAD_ACCUM}')
print(f'Candidates:   {GAMMA_CANDIDATES}')
print()

print(f'{"gamma":>8}  {"PPL":>8}  {"R_bar":>8}  {"gamma_geo":>10}  {"excl%":>6}')
print('-' * 50)
for g in sorted(geodesic_results.keys()):
    gr = geodesic_results[g]
    print(f'{g:8.3f}  {gr["val_ppl"]:8.2f}  {gr["R_bar_mean"]:8.4f}  '
          f'{gr["gamma_geo_mean"]:10.4f}  {gr["excluded_frac"]:6.2%}')

best_g = min(geodesic_results.keys(),
             key=lambda g: geodesic_results[g]['val_ppl'] if geodesic_results[g]['val_ppl'] else 1e9)
print(f'\nBest PPL:     gamma={best_g:.3f}  PPL={geodesic_results[best_g]["val_ppl"]:.2f}')

best_geo = min(geodesic_results.keys(),
               key=lambda g: geodesic_results[g]['R_bar_mean'])
print(f'Best R_bar:   gamma={best_geo:.3f}  R_bar={geodesic_results[best_geo]["R_bar_mean"]:.4f}')

print(f'\nOutputs saved to: {SWEEP_OUTPUT_DIR}')
print('Done!')